In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("zalando-research/fashionmnist")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [ ]:
# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
train_df = pd.read_csv('/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_train.csv')
train_df.head()

In [ ]:
test_df = pd.read_csv('/kaggle/input/datasets/organizations/zalando-research/fashionmnist/fashion-mnist_test.csv')

In [ ]:
class_distribution = train_df.iloc[:, 0].value_counts()
print(class_distribution)

In [ ]:
class_percentages = train_df.iloc[:, 0].value_counts() / len(train_df)
print(class_percentages)

In [ ]:
X_train_full = train_df.iloc[:, 1:].values
y_train_full = train_df.iloc[:, 0].values

X_test = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values


X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=42
)

In [ ]:

# Calculate percentages as Pandas Series
train_pct = pd.Series(y_train).value_counts(normalize=True) * 100
val_pct = pd.Series(y_val).value_counts(normalize=True) * 100

# Combine into a single DataFrame for easy comparison
df_comparison = pd.DataFrame({
    'Train %': train_pct,
    'Validation %': val_pct
}).sort_index()

print(df_comparison)

In [ ]:
X_train_full = train_df.iloc[:, 1:].values
y_train_full = train_df.iloc[:, 0].values

X_test = test_df.iloc[:, 1:].values
y_test = test_df.iloc[:, 0].values


X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, 
    y_train_full, 
    test_size=0.1, 
    random_state=42,
    stratify=y_train_full # <--- This enforces the equal class distribution
)

In [ ]:
import pandas as pd

# Calculate percentages as Pandas Series
train_pct = pd.Series(y_train).value_counts(normalize=True) * 100
val_pct = pd.Series(y_val).value_counts(normalize=True) * 100

# Combine into a single DataFrame for easy comparison
df_comparison = pd.DataFrame({
    'Train %': train_pct,
    'Validation %': val_pct
}).sort_index()

print(df_comparison)

In [ ]:
from torchvision.transforms import transforms

custom_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(), # conversion of tensor + scaling
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
from PIL import Image 
import numpy as np

class CustomDataset(Dataset):
    def __init__(self, features, labels, transform):
        self.features = features
        self.labels = labels 
        self.transform = transform

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        image = self.features[index].reshape(28, 28)
        image = image.astype(np.uint8)
        image = np.stack([image] * 3, axis = -1) # axis = -1 converts C, H, W to H, W, C which PLI format expects
        image = Image.fromarray(image)
        image = self.transform(image)
        return image, torch.tensor(self.labels[index], dtype = torch.long)
        

In [ ]:
train_dataset = CustomDataset(X_train, y_train, transform = custom_transform)
val_dataset = CustomDataset(X_val, y_val, transform = custom_transform)   
test_dataset = CustomDataset(X_test, y_test, transform = custom_transform)

In [ ]:
import torchvision.models as models

vgg16 = models.vgg16(weights = models.VGG16_Weights.DEFAULT)

In [ ]:
vgg16

In [ ]:
for param in vgg16.features.parameters():
    param.requires_grad = False

In [ ]:
!pip install mlflow --quiet

In [ ]:
import mlflow
import mlflow.pytorch

In [ ]:
mlflow.set_tracking_uri("sqlite:////kaggle/working/mlflowVgg16.db")
mlflow.set_experiment("Vgg16_Optuna")

In [ ]:
class VGGTransferModel(nn.Module):
    def __init__(self, base_model, num_fc_layers, fc_neurons_list, dropout_rate, num_classes):
        super().__init__()
        self.features = base_model.features
        self.avgpool = base_model.avgpool

        fc_layers = [nn.Flatten()]
        fc_input = self.getFlattenedSize()
        
        for neurons in fc_neurons_list:
            fc_layers.append(nn.Linear(fc_input, neurons))
            fc_layers.append(nn.BatchNorm1d(neurons))
            fc_layers.append(nn.ReLU())
            fc_layers.append(nn.Dropout(p = dropout_rate))
            fc_input = neurons

        fc_layers.append(nn.Linear(fc_input, num_classes))
        self.classifier = nn.Sequential(*fc_layers)

    def getFlattenedSize(self):
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 224, 224)
            x = self.features(dummy_input)
            x = self.avgpool(x)
            return x.view(1, -1).shape[1]

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

In [ ]:
import optuna

In [ ]:
def objective(trial):
    with mlflow.start_run():
        try:
            num_fc_layers = trial.suggest_int("num_fc_layers", 2, 5)
            fc_neurons_list = [trial.suggest_int(f"fc_neurons_{i}", 64, 512, step = 64) for i in range(num_fc_layers)]
            dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step = 0.1)
            learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-2, log = True)
            batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])
            optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD", "RMSprop"])
            weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log = True)
            epochs = trial.suggest_int("epochs", 10, 30, step = 5) 
    
            mlflow.log_params(trial.params)
    
            train_loader = DataLoader(train_dataset, batch_size = batch_size, shuffle = True, pin_memory = True, num_workers = 4)
            val_loader   = DataLoader(val_dataset, batch_size = batch_size, shuffle = False, pin_memory = True, num_workers = 4)
            test_loader  = DataLoader(test_dataset, batch_size = batch_size, shuffle = False, pin_memory = True, num_workers = 4)

            base_model = models.vgg16(weights = models.VGG16_Weights.DEFAULT)
            for param in base_model.parameters():
                param.requires_grad = False
            
            model = VGGTransferModel(
                base_model = base_model,
                num_fc_layers = num_fc_layers,
                fc_neurons_list = fc_neurons_list,
                dropout_rate = dropout_rate,
                num_classes = 10
            )
            model = model.to(device)
            
            criterion = nn.CrossEntropyLoss()
            trainable_params = filter(lambda p: p.requires_grad, model.parameters())

            if optimizer_name == "Adam":
                optimizer = optim.Adam(trainable_params, lr = learning_rate, weight_decay = weight_decay)
            elif optimizer_name == "SGD":
                optimizer = optim.SGD(trainable_params, lr = learning_rate, weight_decay = weight_decay)
            else:
                optimizer = optim.RMSprop(trainable_params, lr = learning_rate, weight_decay = weight_decay)


            for epoch in range(epochs):
                model.train()
                total_train_loss = 0
                for batch_features, batch_labels in train_loader:
                    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                    outputs = model(batch_features)
                    loss = criterion(outputs, batch_labels)
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()
                    total_train_loss += loss.item()
                avg_train_loss = total_train_loss / len(train_loader)

                model.eval()
                val_correct, val_total, total_val_loss = 0, 0, 0
                with torch.no_grad():
                    for batch_features, batch_labels in val_loader:
                        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                        outputs = model(batch_features)
                        loss = criterion(outputs, batch_labels)
                        total_val_loss += loss.item()
                        _, predicted = torch.max(outputs, 1)
                        val_total += batch_labels.size(0)
                        val_correct += (predicted == batch_labels).sum().item()
                avg_val_loss = total_val_loss / len(val_loader)
                val_accuracy = val_correct / val_total
                mlflow.log_metrics({
                    "train_loss": avg_train_loss,
                    "val_loss": avg_val_loss,
                    "val_accuracy": val_accuracy
                }, step = epoch)

            model.eval()
            total, correct = 0, 0
            with torch.no_grad():
                for batch_features, batch_labels in test_loader:
                    batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
                    outputs = model(batch_features)
                    _, predicted = torch.max(outputs, 1)
                    total += batch_labels.size(0)
                    correct += (predicted == batch_labels).sum().item()
            accuracy = correct / total
            mlflow.log_metric("test_accuracy", accuracy)
    
            example_input = torch.zeros(1, 3, 224, 224)
            mlflow.pytorch.log_model(model, name = "VGG16_transfer_model", input_example = example_input, serialization_format = "pickle")
    
            return accuracy
        except RuntimeError as e:
            mlflow.log_param("error", str(e))
            raise optuna.exceptions.TrialPruned()

In [ ]:
study = optuna.create_study(
    direction = 'maximize',
    study_name = 'VGG16_transfer_model_optuna',
    storage = 'sqlite:////kaggle/working/VGG16_transfer_model_optuna.db',
    load_if_exists = True
)

In [ ]:
study.optimize(objective, n_trials = 20)